# Regime Model Comparison — HMM vs GMM × 1/2/3 Regimes

This notebook studies how **regime detection method** (HMM vs GMM) and **regime granularity** (1, 2, 3) affect risk estimates.

| Config | Method | Regimes | Description |
|--------|--------|---------|-------------|
| Baseline | — | 1 | Global mu/sigma (no regime structure) |
| HMM-2 | Hidden Markov | 2 | Calm / Crisis |
| HMM-3 | Hidden Markov | 3 | Calm / Moderate / Crisis |
| GMM-2 | Gaussian Mixture + RF | 2 | Calm / Crisis |
| GMM-3 | Gaussian Mixture + RF | 3 | Calm / Moderate / Crisis |

Key question: **Does adding regime structure meaningfully change risk estimates? Does going from 2→3 regimes add signal or just noise?**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from src.data.fetch import fetch_asset_data
from src.data.process import clean_market_data, add_returns
from src.analytics.monte_carlo import simulate_paths, simulation_summary, compute_var, compute_cvar
from src.analytics.regime_hmm import fit_hmm, predict_current_regime
from src.analytics.regime_gmm import fit_gmm, predict_current_regime as gmm_predict_current_regime

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Data & Configuration

In [ ]:
TICKER = "BTC-USD"
N_DAYS = 252
N_SIMS = 10_000
SEED = 42

raw = fetch_asset_data(TICKER)
df = clean_market_data(raw)
df = add_returns(df)
returns = df["returns"]
close = df["close"]
initial_price = close.iloc[-1]

print(f"Loaded {len(df)} trading days for {TICKER}")
print(f"Simulation: {N_SIMS:,} paths × {N_DAYS} days, seed={SEED}")
print(f"Initial price: ${initial_price:,.0f}")

In [ ]:
# Fit all 6 configurations
configs = {}
for n in [1, 2, 3]:
    hmm_result = fit_hmm(returns, n_regimes=n, seed=SEED)
    gmm_result = fit_gmm(returns, n_regimes=n, seed=SEED)
    configs[f"HMM-{n}"] = {"type": "hmm", "n": n, "result": hmm_result}
    configs[f"GMM-{n}"] = {"type": "gmm", "n": n, "result": gmm_result}

print("Fitted all 6 configurations.")

## 2. Regime Assignments

Side-by-side price charts with regime coloring. Each row = one regime count (2 or 3). Left = HMM, Right = GMM.

In [ ]:
regime_colors = {0: "#2ca02c", 1: "#ff7f0e", 2: "#d62728"}
regime_names_map = {2: {0: "Calm", 1: "Crisis"}, 3: {0: "Calm", 1: "Moderate", 2: "Crisis"}}

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)

for row, n in enumerate([2, 3]):
    for col, method in enumerate(["HMM", "GMM"]):
        ax = axes[row, col]
        cfg = configs[f"{method}-{n}"]
        labels = cfg["result"]["regime_labels"]

        # Align dates: HMM labels align to full returns, GMM to features_index
        if method == "GMM" and n > 1:
            dates = cfg["result"]["features_index"]
            prices = close.loc[dates].values
        else:
            dates = close.index[:len(labels)]
            prices = close.values[:len(labels)]

        ax.plot(dates, prices, color="steelblue", linewidth=0.6, alpha=0.8)
        for r in range(n):
            mask = labels == r
            name = regime_names_map[n].get(r, f"R{r}")
            ax.fill_between(dates, prices.min(), prices.max(),
                            where=mask, alpha=0.15, color=regime_colors[r], label=name)
        ax.set_title(f"{method} — {n} regimes")
        ax.legend(loc="upper left", fontsize=8)
        ax.set_ylabel("Price (USD)")

axes[1, 0].set_xlabel("Date")
axes[1, 1].set_xlabel("Date")
fig.suptitle(f"{TICKER} — Regime Assignments: HMM vs GMM", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Per-Regime Parameters

Drift (μ) and volatility (σ) for each regime across all 6 configurations. Observation count shows how much data each regime captures.

In [ ]:
rows = []
for name, cfg in configs.items():
    labels = cfg["result"]["regime_labels"]
    for i, p in enumerate(cfg["result"]["regime_params"]):
        obs_count = int((labels == i).sum())
        regime_label = regime_names_map.get(cfg["n"], {}).get(i, f"R{i}") if cfg["n"] > 1 else "Global"
        rows.append({
            "Config": name,
            "Regime": regime_label,
            "μ (daily)": f"{p['mu']:.6f}",
            "σ (daily)": f"{p['sigma']:.4f}",
            "σ (ann.)": f"{p['sigma'] * np.sqrt(252):.2%}",
            "Observations": obs_count,
            "% of data": f"{obs_count / len(labels):.1%}",
        })

params_df = pd.DataFrame(rows)
params_df

## 4. Transition Matrices

How likely is each regime to persist or switch? HMM learns this directly; for GMM we estimate from observed label sequences.

In [ ]:
def observed_transition_matrix(labels, n_regimes):
    """Estimate transition matrix from label sequence."""
    counts = np.zeros((n_regimes, n_regimes))
    for i in range(len(labels) - 1):
        counts[labels[i], labels[i + 1]] += 1
    row_sums = counts.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    return counts / row_sums

for n in [2, 3]:
    names = regime_names_map[n]
    idx = [names[i] for i in range(n)]

    hmm_trans = configs[f"HMM-{n}"]["result"]["transition_matrix"]
    gmm_labels = configs[f"GMM-{n}"]["result"]["regime_labels"]
    gmm_trans = observed_transition_matrix(gmm_labels, n)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, mat, method in zip(axes, [hmm_trans, gmm_trans], ["HMM", "GMM"]):
        sns.heatmap(mat, annot=True, fmt=".2%", cmap="YlOrRd", xticklabels=idx,
                    yticklabels=idx, ax=ax, vmin=0, vmax=1, cbar=False)
        ax.set_title(f"{method} — {n} regimes")
        ax.set_xlabel("To")
        ax.set_ylabel("From")

    fig.suptitle(f"Transition Matrices ({n} regimes)", fontsize=13)
    plt.tight_layout()
    plt.show()

## 5. Monte Carlo Simulation — All 6 Configs

Run `simulate_paths()` for each configuration and collect risk metrics.

In [ ]:
mc_results = {}

for name, cfg in configs.items():
    vol_model = cfg["type"]
    param_key = "hmm_params" if vol_model == "hmm" else "gmm_params"

    paths = simulate_paths(
        close, returns, n_days=N_DAYS, n_simulations=N_SIMS, seed=SEED,
        volatility_model=vol_model, **{param_key: cfg["result"]},
    )
    fp = paths.iloc[-1]
    s95 = simulation_summary(fp, initial_price, confidence=0.95)
    s99 = simulation_summary(fp, initial_price, confidence=0.99)

    mc_results[name] = {
        "paths": paths,
        "final_prices": fp,
        "var_95": s95["var"], "cvar_95": s95["cvar"],
        "var_99": s99["var"], "cvar_99": s99["cvar"],
        "prob_gain": s95["prob_gain"],
        "mean_price": s95["mean_final_price"],
        "median_price": s95["median_final_price"],
    }

print("All 6 simulations complete.")

In [ ]:
# Summary table
summary_rows = []
for name, r in mc_results.items():
    summary_rows.append({
        "Config": name,
        "VaR 95%": f"{r['var_95']:.2%}",
        "CVaR 95%": f"{r['cvar_95']:.2%}",
        "VaR 99%": f"{r['var_99']:.2%}",
        "CVaR 99%": f"{r['cvar_99']:.2%}",
        "P(Gain)": f"{r['prob_gain']:.1%}",
        "Mean Price": f"${r['mean_price']:,.0f}",
        "Median Price": f"${r['median_price']:,.0f}",
    })

summary_df = pd.DataFrame(summary_rows).set_index("Config")
summary_df

## 6. VaR & CVaR Comparison

Bar charts showing tail risk at 95% and 99% confidence across all configurations.

In [ ]:
model_names = list(mc_results.keys())
x = np.arange(len(model_names))
width = 0.35

for conf, var_key, cvar_key in [("95%", "var_95", "cvar_95"), ("99%", "var_99", "cvar_99")]:
    vars_ = [mc_results[m][var_key] for m in model_names]
    cvars_ = [mc_results[m][cvar_key] for m in model_names]

    fig, ax = plt.subplots(figsize=(12, 5))
    bars1 = ax.bar(x - width/2, vars_, width, label=f"VaR ({conf})", color="#ff7f0e")
    bars2 = ax.bar(x + width/2, cvars_, width, label=f"CVaR ({conf})", color="#d62728")

    ax.set_ylabel("Return (%)")
    ax.set_title(f"{TICKER} — VaR & CVaR {conf} by Regime Config ({N_DAYS} days)")
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=20, ha="right")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.legend()
    ax.axhline(0, color="gray", linewidth=0.5)

    for bars in [bars1, bars2]:
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.02,
                    f"{bar.get_height():.1%}", ha="center", va="top", fontsize=8, fontweight="bold")

    plt.tight_layout()
    plt.show()

## 7. Final Price Distributions

KDE overlay of all 6 configurations to see how regime structure reshapes the distribution.

In [ ]:
colors = {"HMM-1": "#1f77b4", "HMM-2": "#ff7f0e", "HMM-3": "#d62728",
          "GMM-1": "#1f77b4", "GMM-2": "#2ca02c", "GMM-3": "#9467bd"}
linestyles = {"HMM-1": "-", "HMM-2": "-", "HMM-3": "-",
              "GMM-1": "--", "GMM-2": "--", "GMM-3": "--"}

fig, ax = plt.subplots(figsize=(14, 5))
for name, r in mc_results.items():
    sns.kdeplot(r["final_prices"], ax=ax, label=name, color=colors[name],
                linestyle=linestyles[name], linewidth=1.5)
ax.axvline(initial_price, color="gray", linewidth=1.5, linestyle=":", label=f"Initial: ${initial_price:,.0f}")
ax.set_title(f"{TICKER} — Final Price Distribution by Regime Config")
ax.set_xlabel("Price (USD)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Path Fan Plots

200 sample paths per config. 3×2 grid: rows = regime count (1, 2, 3), columns = method (HMM, GMM).

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14), sharex=True, sharey=True)

for row, n in enumerate([1, 2, 3]):
    for col, method in enumerate(["HMM", "GMM"]):
        ax = axes[row, col]
        name = f"{method}-{n}"
        paths = mc_results[name]["paths"]
        sample = paths.iloc[:, :200]

        ax.plot(sample, color=colors[name], alpha=0.03, linewidth=0.5)
        ax.plot(paths.median(axis=1), color="white", linewidth=1.5, label="Median")
        ax.plot(paths.quantile(0.05, axis=1), color="black", linewidth=1, linestyle="--", label="5th pctl")
        ax.plot(paths.quantile(0.95, axis=1), color="black", linewidth=1, linestyle="--", label="95th pctl")
        ax.axhline(initial_price, color="gray", linewidth=0.8, linestyle=":")
        ax.set_title(f"{name}")
        ax.legend(loc="upper left", fontsize=7)

for ax in axes[-1]:
    ax.set_xlabel("Trading Day")
for ax in axes[:, 0]:
    ax.set_ylabel("Price (USD)")

fig.suptitle(f"{TICKER} — Monte Carlo Paths by Regime Config ({N_SIMS:,} sims, {N_DAYS} days)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 9. Regime Count Study

How do risk estimates change as regime granularity increases (1 → 2 → 3)? Line plots for VaR and CVaR at both confidence levels.

In [ ]:
regime_counts = [1, 2, 3]
metrics = ["var_95", "cvar_95", "var_99", "cvar_99"]
metric_labels = {"var_95": "VaR 95%", "cvar_95": "CVaR 95%", "var_99": "VaR 99%", "cvar_99": "CVaR 99%"}

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=False)

for ax, metric in zip(axes.flat, metrics):
    for method, marker, ls in [("HMM", "o", "-"), ("GMM", "s", "--")]:
        vals = [mc_results[f"{method}-{n}"][metric] for n in regime_counts]
        ax.plot(regime_counts, vals, marker=marker, linestyle=ls, linewidth=2,
                markersize=8, label=method)
        for n, v in zip(regime_counts, vals):
            ax.annotate(f"{v:.1%}", (n, v), textcoords="offset points",
                        xytext=(10, 5), fontsize=8)

    ax.set_title(metric_labels[metric])
    ax.set_xlabel("Number of Regimes")
    ax.set_ylabel("Return (%)")
    ax.set_xticks(regime_counts)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle(f"{TICKER} — Risk Metrics vs Regime Granularity", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 10. Key Takeaways

- **1 regime (baseline):** equivalent to constant vol GBM — no regime structure. Serves as the control.
- **1 → 2 regimes:** the biggest jump in risk differentiation. Both HMM and GMM detect a calm vs crisis split. Crisis regime has higher σ and often negative μ, pulling VaR/CVaR deeper. This is the most significant structural upgrade.
- **2 → 3 regimes:** adds a moderate regime between calm and crisis. Whether this adds meaningful signal depends on the asset — for volatile assets like BTC, a third regime may capture transition periods; for stable assets, it may just fragment the calm regime.
- **HMM vs GMM:** HMM learns transition dynamics directly (Markov property); GMM clusters on engineered features and approximates transitions from observed sequences. HMM tends to produce smoother regime boundaries; GMM can be noisier but captures richer feature interactions.
- **Practical recommendation:** 2 regimes is the sweet spot for most use cases — it captures the calm/crisis asymmetry that drives tail risk without overfitting. Use 3 only if the regime parameter table shows a genuinely distinct middle regime (not just a fragmented version of calm).